In [1]:
!pip install tqdm


[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from data.loader import *

from evaluation.ground_truth import *
from evaluation.ranking import *
from evaluation.metrics import *

from features.matrices import  build_association_dict 
from features.pairDrugCancerGuney import* 

from graph.diffusion import *

from models.GuneyModel import compute_proximity_matrix_fast

from models.nmf import *

from utils.io import *
from utils.logger import*
from utils.model_io import *
from utils.pickle import *
import numpy as np

In [3]:

paths = {
    "ppi": "data\\dadosPreProcessamento\\DadosIC- PPI_gysi.csv",
    "drug_targets": 'data\\dadosPreProcessamento\\DadosIC- drug_targets.csv',
    "cancer": 'data\\dadosPreProcessamento\\DadosICcancer_filtrado.csv',
    "repo": "data\\dadosPreProcessamento\\repoDB.csv"
}

In [4]:
ppi, drug_targets, cancer, repo = load_data(paths)

In [5]:
G = build_graph(ppi)

Construindo um dicionario de associações chave-valor, onde a chave é a identificação do medicamento e o valor é uma lista dos targets dessa droga na ppi humana.

In [6]:
P_d = build_association_dict(drug_targets, "drugbank_id", "entrez_id")
P_c = build_association_dict(cancer, "diseaseid", "geneid")

In [7]:
truth = build_ground_truth(repo, status_filter="approved")

In [ ]:
# Calcule as matrizes 
Z_df, D_df = compute_proximity_matrix_fast(G, P_d, P_c, n_random=100, seed=452456, verbose=True)

Pré-calculando matriz de distâncias...
Matriz de distâncias criada: (18505, 18505)
Construindo mapeamento de graus...
Total de pares a processar: 124501
Iniciando processamento com 4 workers...


Processando pares:   0%|          | 32/124501 [04:34<681:59:16, 19.73s/it]

In [ ]:
# Construir ranking (apenas os proximais, se desejar)
rankings = build_ranking_dict(Z_df, proximal_threshold=-0.15)#threshold de -0.15 é usado pelo Guney em seu artigo

In [ ]:
# Avaliar
recall_ks = [10,20,50]
recall_dict = {}

aucs = auc_per_cancer_from_Z(Z_df, truth)

for K in recall_ks:
    recall_dict[K] = recall_at_k(
        rankings,
        truth,
        K
    )

In [ ]:
"""
pseudocódigo:
carregar dados(ppi,drugatargets,cancer,repoDB);
gerar grafo com ppi;
Construir associações
gerar combinações de par droga-doença;
para cada par droga-doença calcular proximidade 
colocar o valor de retorno da função que calcula proximidade numa matriz(Z)/dataframe
para cada doença ordenar do maior para o menor o valor de proximidade z e filtrar os que são proximal  
colocar essa lista ordenada e filtrada de canda cancer num dicionario chamado rankng.
calcular auc por cancer usando a matriz Z e o ground truth do RepoDB
calcular recall at top k usando o ranking
"""
